# Django: отладка и профилирование

### Предметная область

**Книжный клуб** — сообщество читателей, которые читают книги, оставляют рецензии и ведут личный список чтения.

#### Кто участвует и что делает

```
 ┌─────────────────────────────────────────────────────┐
 │                   Участник (Member)                  │
 │                                                      │
 │  • присоединяется к клубу                           │
 │  • читает книги                                      │
 │  • пишет рецензии с оценкой от 1 до 5               │
 │  • ведёт список чтения (хочу / читаю / прочитал)    │
 └──────────────┬───────────────────┬───────────────────┘
                │                   │
          пишет │             ведёт │
                ▼                   ▼
 ┌──────────────────────┐  ┌────────────────────────┐
 │   Рецензия (Review)  │  │  Список чтения         │
 │                      │  │  (ReadingList)         │
 │  • оценка (1–5)      │  │                        │
 │  • текст             │  │  • статус:             │
 │  • дата              │  │    хочу прочитать      │
 └──────────┬───────────┘  │    читаю сейчас        │
            │              │    прочитал(а)         │
  оценивает │              └───────────┬────────────┘
            │                          │
            └──────────┬───────────────┘
                       │ ссылается на
                       ▼
          ┌────────────────────────┐
          │      Книга (Book)      │
          │                        │
          │  • название, автор     │
          │  • год издания         │
          │  • описание            │
          └────────────┬───────────┘
                       │ относится к
                       ▼
          ┌────────────────────────┐
          │      Жанр (Genre)      │
          │  • название            │
          │  • slug (для URL)      │
          └────────────────────────┘
```

#### Ключевые правила

- Один участник может написать **только одну рецензию** на каждую книгу
- Одна книга может быть **у многих участников** в списке чтения одновременно
- `Member` не заменяет встроенного `User` Django, а **расширяет его** через `OneToOneField` — стандартный паттерн
- `Genre` вынесен в отдельную модель, чтобы не хранить строку жанра в каждой книге — это источник задачи `select_related`


## О проекте: Книжный клуб

Репозиторий: https://github.com/vitaly-efremov/django-bookclub

Проект намеренно содержит несколько мест с неоптимальными запросами к БД: их мы будем находить, объяснять причину и исправлять.

### Что реализовано

| Страница | URL | Описание |
|----------|-----|----------|
| Главная | `/` | Список книг с жанрами |
| Книга | `/books/<id>/` | Детальная страница с рецензиями и средней оценкой |
| Рецензии | `/reviews/` | Все рецензии участников |
| Профиль | `/members/<username>/` | Рецензии и список чтения участника |
| API книг | `/api/books/` | JSON: список книг с фильтрацией и поиском |
| API жанров | `/api/genres/` | JSON: жанры с количеством книг |

### Модели и связи между ними

```
User (встроенный Django) ──1:1── Member
                                    │
Genre ──FK── Book ──FK──────── Review ──FK── Member
                │
                └──FK── ReadingList ──FK── Member
```

- **Genre** — жанр книги (`fiction`, `classic` и др.)
- **Book** — книга, связана с одним жанром через `ForeignKey`
- **Member** — участник клуба, расширяет встроенного `User` через `OneToOneField`
- **Review** — рецензия: связывает `Book` и `Member`, содержит оценку (1–5) и текст
- **ReadingList** — запись в списке чтения: связывает `Member` и `Book`, хранит статус (`want` / `reading` / `done`)

### Структура проекта

```
bookclub/
├── bookclub/            # конфигурация проекта
│   ├── settings/
│   │   ├── base.py      # общие настройки
│   │   ├── dev.py       # разработка: Debug Toolbar, SQL-логирование
│   │   └── prod.py      # production: DEBUG=False, SECRET_KEY из env
│   └── urls.py
├── clubs/               # основное приложение
│   ├── models.py        # Genre, Book, Member, Review, ReadingList
│   ├── views.py         # страницы и JSON API (с намеренными N+1)
│   ├── admin.py
│   └── tests.py
├── templates/           # HTML-шаблоны
├── pyproject.toml       # зависимости Poetry
└── manage.py
```

### Быстрый старт

```bash
git clone https://github.com/vitaly-efremov/django-bookclub
cd django-bookclub
poetry install
poetry run python manage.py migrate
poetry run python manage.py seed    # заполнить БД тестовыми данными
poetry run python manage.py runserver
```

Admin-панель: http://127.0.0.1:8000/admin (логин: `admin`, пароль: `admin123`)


## 1. Почему Django-приложения тормозят

В большинстве веб-приложений узкое место — не Python-код, а **база данных**. Время обработки HTTP-запроса складывается из:
- времени выполнения SQL-запросов,
- времени Python-вычислений,
- времени рендеринга шаблонов.

На практике 80–95% времени запроса — это работа с БД. Поэтому профилирование Django-приложения начинается с анализа SQL.

### Откуда берётся медлительность

Три основные причины:

1. **Много запросов** — вместо одного JOIN Django делает десятки отдельных SELECT. Это называется проблемой N+1.
2. **Тяжёлые запросы** — поиск по неиндексированному полю, полный скан таблицы вместо индексного поиска.
3. **Вычисления в Python вместо SQL** — загружаем все строки в память и считаем в цикле то, что СУБД умеет делать за один запрос.

### Типичные проблемы ORM

| Проблема | Что происходит |
|----------|----------------|
| `Book.objects.all()` без `select_related` | Для каждой книги — отдельный SELECT за жанром |
| `sum(r.rating for r in book.reviews.all())` | Загружает все рецензии в Python, считает в цикле |
| `filter(title__icontains='война')` без индекса | PostgreSQL читает всю таблицу |

Чтобы найти эти проблемы, нужен инструмент, который показывает SQL-запросы прямо во время работы приложения.

## 2. Управление зависимостями: Poetry

### Зачем нужен менеджер зависимостей

Когда проект использует несколько библиотек, возникают задачи:
- зафиксировать точные версии, чтобы у всех участников команды была одна среда;
- отделить зависимости проекта от глобального Python;
- легко воспроизвести окружение на новой машине или сервере.

Стандартный инструмент — `venv` + `pip` + `requirements.txt`. Это работает, но неудобно:
- `requirements.txt` хранит только прямые зависимости **или** полный список с транзитивными — приходится выбирать;
- нет встроенного механизма разделения на dev/prod зависимости;
- нет lock-файла с гарантированно воспроизводимыми версиями.

### Основная идея Poetry

**Poetry** решает эти проблемы двумя файлами:

| Файл | Что хранит | Кто редактирует |
|------|-----------|----------------|
| `pyproject.toml` | зависимости с ограничениями версий (`django = "^6.0"`) | разработчик вручную |
| `poetry.lock` | точные версии всех пакетов (включая транзитивные) | Poetry автоматически |

`poetry.lock` коммитится в git — это гарантирует, что `poetry install` на любой машине поставит **ровно те же версии**.

Poetry также автоматически создаёт и управляет виртуальным окружением — не нужно делать `python -m venv .venv` вручную.

### Преимущества перед venv + pip

| Критерий | venv + pip | Poetry |
|----------|-----------|-------|
| Воспроизводимость | `requirements.txt` — нет гарантий | `poetry.lock` — точные версии |
| Виртуальное окружение | создаётся вручную | создаётся автоматически |
| Dev-зависимости | один файл на всё | `[tool.poetry.dev-dependencies]` |
| Разрешение конфликтов | нет | встроенный solver |
| Публикация пакета | отдельные инструменты | встроено |

### Основные операции

**Установить зависимости из lock-файла** (первый запуск на новой машине):
```bash
poetry install
```

**Запустить команду в окружении проекта:**
```bash
poetry run python manage.py runserver
poetry run python manage.py migrate
poetry run pytest
```

**Активировать окружение в текущем терминале** (аналог `source .venv/bin/activate`):
```bash
poetry shell
```

**Добавить новую зависимость:**
```bash
poetry add requests          # production
poetry add --group dev pytest  # только для разработки
```

**Удалить зависимость:**
```bash
poetry remove requests
```

**Обновить зависимости:**
```bash
poetry update          # все пакеты в рамках ограничений
poetry update django   # конкретный пакет
```

**Посмотреть установленные пакеты:**
```bash
poetry show
poetry show --tree     # с транзитивными зависимостями
```

### В проекте bookclub

Проект использует Poetry. Зависимости описаны в `pyproject.toml`:

```toml
[tool.poetry.dependencies]
python = "^3.12"
django = "6.0.5"
psycopg2-binary = "2.9.12"
django-debug-toolbar = "6.3.0"
pytest-django = "4.12.0"
faker = "40.15.0"
```

Чтобы начать работу с проектом на новой машине, достаточно:
```bash
git clone https://github.com/vitaly-efremov/django-bookclub
cd django-bookclub
poetry install
poetry run python manage.py migrate
poetry run python manage.py seed
poetry run python manage.py runserver
```

## 3. Разделение настроек: dev и prod

В реальном проекте настройки для разработки и production сильно отличаются:
- в dev нужен `DEBUG=True`, debug_toolbar, SQL-логирование;
- в prod `DEBUG` должен быть `False`, секретный ключ — из переменной окружения, никаких инструментов отладки.

Хранить всё в одном `settings.py` неудобно: приходится вручную менять значения или комментировать блоки. Стандартное решение — разбить настройки на несколько файлов.

### Структура

```
bookclub/settings/
├── __init__.py
├── base.py   — общие настройки (БД, приложения, шаблоны, локализация)
├── dev.py    — для локальной разработки
└── prod.py   — для production-сервера
```

`dev.py` и `prod.py` импортируют всё из `base.py` и переопределяют только то, что отличается.

### base.py — общая основа

```python
# bookclub/settings/base.py
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent.parent

INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    # ... стандартные приложения
    'clubs',
]

DATABASES = {
    'default': {
        'ENGINE': 'django.db.backends.postgresql',
        'NAME': 'bookclub',
        # ...
    }
}
```

### dev.py — настройки для разработки

```python
# bookclub/settings/dev.py
from .base import *

DEBUG = True
ALLOWED_HOSTS = ['localhost', '127.0.0.1']

# Debug Toolbar — только в dev
INSTALLED_APPS += ['debug_toolbar']
MIDDLEWARE = ['debug_toolbar.middleware.DebugToolbarMiddleware'] + MIDDLEWARE
INTERNAL_IPS = ['127.0.0.1']

# SQL-логирование — только в dev
LOGGING = {
    'version': 1,
    'disable_existing_loggers': False,
    'handlers': {'console': {'class': 'logging.StreamHandler'}},
    'loggers': {
        'django.db.backends': {'handlers': ['console'], 'level': 'DEBUG'},
    },
}
```

### prod.py — настройки для сервера

```python
# bookclub/settings/prod.py
from .base import *
import os

DEBUG = False
ALLOWED_HOSTS = []  # указать реальный домен

# Секретный ключ никогда не хранится в коде
SECRET_KEY = os.environ['DJANGO_SECRET_KEY']
```

### Выбор файла настроек

Django узнаёт, какой файл использовать, через переменную окружения `DJANGO_SETTINGS_MODULE`.

В `manage.py` прописан dev как значение по умолчанию:
```python
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'bookclub.settings.dev')
```

В `wsgi.py` / `asgi.py` — prod:
```python
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'bookclub.settings.prod')
```

Переключить явно при запуске:
```bash
DJANGO_SETTINGS_MODULE=bookclub.settings.prod poetry run python manage.py check
```


## 4. Django Debug Toolbar

**Django Debug Toolbar** — это панель, которая встраивается в браузер и показывает подробную информацию о каждом HTTP-запросе: сколько SQL-запросов выполнилось, сколько времени заняло, какие были дублирующиеся запросы.

Toolbar рисует боковую панель прямо на HTML-странице — только для адресов из списка `INTERNAL_IPS`. В production она не появляется.

### Конфигурация в settings.py

В проекте `bookclub` toolbar уже установлен. Вот как выглядит конфигурация:

```python
INSTALLED_APPS = [
    'django.contrib.admin',
    'django.contrib.auth',
    'django.contrib.contenttypes',
    'django.contrib.sessions',
    'django.contrib.messages',
    'django.contrib.staticfiles',

    # Наше приложение
    'clubs',

    # Инструмент профилирования — показывает SQL-запросы прямо на странице
    'debug_toolbar',
]

MIDDLEWARE = [
    # debug_toolbar должен идти как можно раньше
    'debug_toolbar.middleware.DebugToolbarMiddleware',

    'django.middleware.security.SecurityMiddleware',
    'django.contrib.sessions.middleware.SessionMiddleware',
    # ... остальные middleware
]

# Toolbar показывается только для этих IP-адресов
INTERNAL_IPS = [
    '127.0.0.1',
]
```

### Как читать вкладку SQL

После открытия любой страницы в браузере справа появится панель. На вкладке **SQL** видно:
- **количество запросов** — если их больше 10 на простой странице, скорее всего есть N+1;
- **время каждого запроса** — медленные запросы выделяются;
- **похожие запросы** — toolbar помечает дублирующиеся SELECT, которые отличаются только значением параметра (признак N+1).

Пример того, что увидите на странице списка рецензий (`/reviews/`) без оптимизаций:

```
SQL queries: 61  (1.23 ms)

SELECT * FROM clubs_review                          0.12 ms
SELECT * FROM clubs_member WHERE id = 1             0.04 ms
SELECT * FROM auth_user WHERE id = 1                0.03 ms
SELECT * FROM clubs_book WHERE id = 12              0.04 ms
SELECT * FROM clubs_member WHERE id = 2             0.04 ms   [похожий]
SELECT * FROM auth_user WHERE id = 2                0.03 ms   [похожий]
SELECT * FROM clubs_book WHERE id = 7               0.04 ms   [похожий]
...
```

20 рецензий → 1 + 20×3 = 61 запрос. Это классический N+1.

### Запуск проекта

```bash
# Из папки проекта bookclub
poetry run python manage.py runserver
```

Откройте http://127.0.0.1:8000 — в правом углу страницы появится значок DjDT. Нажмите, чтобы раскрыть панель.

## 5. Проблема N+1 запросов

**N+1** — это ситуация, когда для получения N объектов с их связанными данными выполняется не 1 запрос (с JOIN), а 1 + N запросов.

Django ORM по умолчанию **ленив**: связанные объекты подгружаются отдельным запросом в момент обращения. Это удобно в простых случаях, но катастрофично в цикле.

### Реальный код из проекта

**Функция `index()` в `clubs/views.py`:**

```python
def index(request):
    """Главная страница: список книг с базовой информацией."""
    # ПРОБЛЕМА N+1: получаем книги без genre
    # Для каждой книги в шаблоне будет отдельный SELECT для book.genre
    books = Book.objects.all()[:12]

    return render(request, 'clubs/index.html', {'books': books})
```

В шаблоне используется `{{ book.genre.name }}`. Django при каждом обращении к `book.genre` делает отдельный `SELECT * FROM clubs_genre WHERE id = ?`. 12 книг → 1 + 12 = 13 запросов.

**Функция `review_list()` — самый показательный пример:**

```python
def review_list(request):
    """
    Все рецензии на сайте.

    САМЫЙ ПОКАЗАТЕЛЬНЫЙ N+1:
    Для каждой из N рецензий Django делает:
      - 1 запрос за review.member
      - 1 запрос за review.member.user
      - 1 запрос за review.book
    Итого: 1 (список) + N*3 запросов к БД!
    """
    reviews = Review.objects.all()   # нет select_related вообще

    return render(request, 'clubs/review_list.html', {'reviews': reviews})
```

### select_related — JOIN для ForeignKey и OneToOne

`select_related` говорит Django: «сделай один SQL JOIN вместо N отдельных запросов».

Используется для связей **ForeignKey** и **OneToOneField**, где каждый объект ссылается ровно на один связанный объект.

```python
# Было: 1 + N запросов
books = Book.objects.all()[:12]

# Стало: 1 запрос с JOIN
books = Book.objects.select_related('genre')[:12]
```

SQL, который генерирует `select_related('genre')`:
```sql
SELECT clubs_book.*, clubs_genre.*
FROM clubs_book
LEFT OUTER JOIN clubs_genre ON (clubs_book.genre_id = clubs_genre.id)
LIMIT 12;
```

### prefetch_related — для ManyToMany и обратных FK

`prefetch_related` делает **отдельный запрос** для связанных объектов, а потом соединяет результаты в Python. Используется там, где JOIN невозможен или неэффективен: ManyToMany, обратная сторона ForeignKey.

```python
# Загрузить все книги и сразу все их рецензии (обратный FK)
books = Book.objects.prefetch_related('reviews')[:12]
# Выполнит 2 запроса: один за книгами, один за рецензиями
```

### Исправление index()

```python
def index(request):
    books = Book.objects.select_related('genre')[:12]
    return render(request, 'clubs/index.html', {'books': books})
```

Было 13 запросов → стал 1.

### Исправление review_list()

```python
def review_list(request):
    # Цепочка: review -> member -> user (два уровня вниз через двойное подчёркивание)
    reviews = Review.objects.select_related('member__user', 'book')
    return render(request, 'clubs/review_list.html', {'reviews': reviews})
```

Один запрос с тремя JOIN вместо 1 + N×3.

### Подсчёт запросов через connection.queries

Проверить количество запросов можно прямо в Django shell, не открывая браузер:

```python
# В Django shell: poetry run python manage.py shell

from django.db import connection, reset_queries
from django.conf import settings

# Убедитесь, что DEBUG=True — только тогда запросы сохраняются
print(settings.DEBUG)  # True

reset_queries()

from clubs.models import Review
reviews = list(Review.objects.all())          # без select_related
_ = [r.member.user.username for r in reviews] # обращаемся к связанным объектам

print(f'Запросов: {len(connection.queries)}')
# Запросов: 41  (если 20 рецензий)

reset_queries()
reviews = list(Review.objects.select_related('member__user', 'book'))
_ = [r.member.user.username for r in reviews]

print(f'Запросов: {len(connection.queries)}')
# Запросов: 1
```
### Вывод SQL в реальном времени через logging

`connection.queries` показывает запросы постфактум. Если нужно видеть SQL прямо в момент выполнения — настройте логгер в `settings/dev.py`:

```python
# settings/dev.py
LOGGING = {
    'version': 1,
    'disable_existing_loggers': False,
    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
        },
    },
    'loggers': {
        'django.db.backends': {
            'handlers': ['console'],
            'level': 'DEBUG',
        },
    },
}
```

После этого каждый запрос печатается сразу при обращении к БД — и в shell, и при `runserver`:

```
(0.001) SELECT "clubs_book"."id", "clubs_book"."title", ...
        FROM "clubs_book"; args=()
(0.000) SELECT "clubs_genre"."id", "clubs_genre"."name"
        FROM "clubs_genre" WHERE "clubs_genre"."id" = 1; args=(1,)
(0.000) SELECT "clubs_genre"."id", "clubs_genre"."name"
        FROM "clubs_genre" WHERE "clubs_genre"."id" = 2; args=(2,)  ← N+1 виден сразу
```

Настройка живёт только в `dev.py` — в `prod.py` SQL в консоль не пишется.


### Демонстрация: сколько запросов без оптимизации

Следующая ячейка — чистый Python без Django. Она показывает, что происходит при "ленивой" загрузке данных: каждое обращение к связанному объекту генерирует отдельный вызов.

In [ ]:
# Симуляция N+1 без Django: счётчик вызовов

class FakeDB:
    """Имитирует 'ленивую' загрузку из базы."""
    def __init__(self, name):
        self._name = name
        self._call_count = 0

    def get(self, obj_id):
        self._call_count += 1
        return f'{self._name}#{obj_id}'


genre_db = FakeDB('Genre')

# Допустим, у нас 10 книг с разными genre_id
books = [{'title': f'Book {i}', 'genre_id': i % 3 + 1} for i in range(10)]

# Ленивый вариант — для каждой книги отдельный «запрос»
result = []
for book in books:
    genre_name = genre_db.get(book['genre_id'])  # отдельный вызов!
    result.append(f"{book['title']} ({genre_name})")

print(f'Запросов к genre_db: {genre_db._call_count}')  # 10 — по одному на книгу
print('Первые три результата:', result[:3])

In [ ]:
# Оптимизированный вариант — загружаем все жанры одним вызовом

genre_db2 = FakeDB('Genre')

# Один «запрос» за всеми нужными жанрами
needed_ids = {b['genre_id'] for b in books}
genres = {gid: genre_db2.get(gid) for gid in needed_ids}  # 3 вызова вместо 10

result2 = [
    f"{b['title']} ({genres[b['genre_id']]})"
    for b in books
]

print(f'Запросов к genre_db: {genre_db2._call_count}')  # 3 — только уникальные ID
print('Первые три результата:', result2[:3])

## 6. Вычисления в Python vs в базе данных

### Антипаттерн: считаем в цикле

Когда нужно посчитать среднее, сумму или количество, начинающие разработчики часто делают так:
1. Загружают все строки из таблицы в Python.
2. Итерируют по ним в цикле и считают вручную.

Это создаёт лишнюю нагрузку: данные едут по сети из БД в приложение, Python тратит память на объекты, которые нужны только для одного числа.

### Реальный пример: метод average_rating() в Book

Посмотрим на модель из `clubs/models.py`:

```python
def average_rating(self):
    """
    НАМЕРЕННО МЕДЛЕННЫЙ КОД для демонстрации!

    Считаем среднюю оценку в Python, загружая ВСЕ рецензии в память.
    Правильный способ: использовать аннотацию на уровне QuerySet
    через Avg() — тогда вычисление происходит в PostgreSQL.
    """
    reviews = self.reviews.all()   # SELECT * FROM reviews WHERE book_id = ?
    if not reviews:
        return None
    total = sum(r.rating for r in reviews)  # итерируем объекты Python
    return round(total / len(reviews), 1)
```

Что происходит в SQL:
```sql
-- Загружаем ВСЕ строки рецензий для книги
SELECT id, book_id, member_id, rating, text, created_at
FROM clubs_review
WHERE book_id = 42;
-- Возвращает 500 строк → Python получает 500 объектов → считает сам
```

### Правильный подход: aggregate()

Пусть PostgreSQL считает среднее — он умеет это делать за один проход по индексу:

```python
def average_rating_v2(self):
    result = self.reviews.aggregate(avg=Avg('rating'))['avg']
    return round(result, 1) if result is not None else None
```

SQL, который генерирует `aggregate(avg=Avg('rating'))`:
```sql
-- PostgreSQL делает вычисление сам
SELECT AVG(rating) FROM clubs_review WHERE book_id = 42;
-- Возвращает одно число: 4.2
```

Оба метода уже написаны в `clubs/models.py` — можно сравнить прямо в shell.

### aggregate() vs annotate()

**`aggregate()`** — возвращает одно словарь с результатом для всего QuerySet:

```python
from django.db.models import Avg
from clubs.models import Review

# Средняя оценка по всем рецензиям в системе
Review.objects.aggregate(avg=Avg('rating'))
# {'avg': 4.1}
```

**`annotate()`** — добавляет вычисляемое поле к каждому объекту в QuerySet:

```python
from django.db.models import Avg
from clubs.models import Book

# Каждая книга получает поле avg_rating
books = Book.objects.annotate(avg=Avg('reviews__rating'))
for book in books:
    print(book.title, book.avg)
# «Война и мир» 4.5
# «Мастер и Маргарита» 4.8
```

SQL для `annotate`:
```sql
SELECT clubs_book.*, AVG(clubs_review.rating) AS avg
FROM clubs_book
LEFT OUTER JOIN clubs_review ON (clubs_book.id = clubs_review.book_id)
GROUP BY clubs_book.id;
```

Один запрос вместо N вызовов `average_rating()` в цикле.

### Сравнение через connection.queries

```python
# В Django shell

from django.db import connection, reset_queries
from django.db.models import Avg
from clubs.models import Book

# Вариант 1: Python-вычисление
reset_queries()
books = list(Book.objects.all()[:10])
ratings = [b.average_rating() for b in books]  # 10 дополнительных запросов!
print('Вариант 1:', len(connection.queries), 'запросов')

# Вариант 2: annotate
reset_queries()
books = list(Book.objects.annotate(avg=Avg('reviews__rating'))[:10])
ratings = [b.avg for b in books]  # данные уже есть — запросов нет
print('Вариант 2:', len(connection.queries), 'запросов')
```

### Демонстрация: aggregate vs Python-цикл

In [2]:
import timeit

# Имитируем «загрузку из БД» и два способа посчитать среднее

ratings = list(range(1, 6)) * 200  # 1000 оценок

def avg_python(data):
    """Python-цикл (аналог average_rating)"""
    total = sum(r for r in data)
    return total / len(data)

import statistics
def avg_builtin(data):
    """Встроенная функция (аналог aggregate — вычисление вне цикла вручную)"""
    return statistics.mean(data)

n = 10_000
t1 = timeit.timeit(lambda: avg_python(ratings), number=n)
t2 = timeit.timeit(lambda: avg_builtin(ratings), number=n)

print(f'Python sum/len:    {t1/n*1000:.4f} мс')
print(f'statistics.mean(): {t2/n*1000:.4f} мс')
print()
print('В реальном Django разница ещё больше:')
print('aggregate() экономит не только время вычисления,')
print('но и передачу данных из БД в Python.')

Python sum/len:    0.0193 мс
statistics.mean(): 0.1080 мс

В реальном Django разница ещё больше:
aggregate() экономит не только время вычисления,
но и передачу данных из БД в Python.


## 7. Поиск и индексы

### Почему icontains работает медленно

Поиск по подстроке с `icontains` транслируется в SQL как `ILIKE '%текст%'` (без учёта регистра) или `LIKE '%текст%'`.

```sql
-- Django: Book.objects.filter(title__icontains='война')
SELECT * FROM clubs_book WHERE title ILIKE '%война%';
```

Проблема: поиск по шаблону вида `%текст%` (с `%` в начале) **не может использовать индекс**. PostgreSQL вынужден прочитать каждую строку таблицы — это называется `Seq Scan` (sequential scan, последовательное сканирование).

При 100 000 книгах каждый поисковый запрос читает все 100 000 строк.


### Индексы в Django

Индекс — структура данных, которая ускоряет поиск по полю. Аналог алфавитного указателя в книге.

Добавить индекс на поле в Django:

```python
class Book(models.Model):
    # db_index=True создаст индекс при следующей миграции
    title = models.CharField('Название', max_length=300, db_index=True)
```

После добавления нужно создать и применить миграцию:

```bash
poetry run python manage.py makemigrations
poetry run python manage.py migrate
```

Когда добавлять индекс:
- на полях, по которым часто фильтруют (`filter(status=...)`),
- на полях, по которым сортируют (`order_by('title')`),
- на внешних ключах (Django добавляет их автоматически).

Индекс замедляет INSERT/UPDATE, но ускоряет SELECT. На таблицах с преобладанием чтения — всегда добавляйте.

### QuerySet.explain() — план запроса PostgreSQL

`explain()` возвращает план выполнения запроса PostgreSQL: как именно СУБД будет искать данные.

```python
# В Django shell
from clubs.models import Book

print(Book.objects.filter(title__icontains='война').explain(verbose=True, analyze=True))
```

**Без индекса** (что вы увидите сейчас):
```
Seq Scan on clubs_book  (cost=0.00..18.50 rows=1 width=...) (actual time=0.015..0.821 rows=2 loops=1)
  Filter: ((title)::text ~~* '%война%'::text)
  Rows Removed by Filter: 98
Planning Time: 0.1 ms
Execution Time: 0.9 ms
```

`Seq Scan` — PostgreSQL прочитал всю таблицу.

**После добавления индекса** на `title` и использования `contains` (без `%` в начале):
```
Index Scan using clubs_book_title_idx on clubs_book  (cost=0.28..8.29 rows=1 width=...)
  Index Cond: ((title)::text ~~ 'война%'::text)
Planning Time: 0.2 ms
Execution Time: 0.05 ms
```

`Index Scan` — PostgreSQL использовал индекс, время выполнения в 18 раз меньше.

### Демонстрация: Seq Scan vs Index Scan (концептуально)

In [3]:
import timeit

# Аналог Seq Scan: линейный поиск по списку
def seq_scan(data, target):
    """Проходим по всем элементам — аналог Seq Scan."""
    return [x for x in data if target in x]

# Аналог Index Scan: поиск по словарю (хэш-таблица)
def index_scan(index, target):
    """Поиск по ключу в словаре — аналог Index Scan."""
    return index.get(target, [])


# Данные: список строк
import random, string
random.seed(42)
words = [''.join(random.choices(string.ascii_lowercase, k=8)) for _ in range(100_000)]
target = words[50_000]  # ищем конкретное слово

# Строим «индекс» (словарь слово → позиции)
index = {}
for i, w in enumerate(words):
    index.setdefault(w, []).append(i)

n = 1000
t_seq = timeit.timeit(lambda: seq_scan(words, target), number=n)
t_idx = timeit.timeit(lambda: index_scan(index, target), number=n)

print(f'Seq Scan (линейный поиск):  {t_seq/n*1000:.3f} мс')
print(f'Index Scan (словарь):       {t_idx/n*1000:.4f} мс')
print(f'Ускорение: ~{t_seq/t_idx:.0f}x')

Seq Scan (линейный поиск):  1.571 мс
Index Scan (словарь):       0.0001 мс
Ускорение: ~22085x


## 8. Чек-лист оптимизации ORM

| Симптом | Как обнаружить | Решение |
|---------|---------------|---------|
| N+1 для FK | Debug Toolbar: много одинаковых запросов | `select_related` |
| N+1 для M2M | Debug Toolbar | `prefetch_related` |
| Вычисления по всем строкам | Большое время запроса, Python-цикл в коде | `aggregate` / `annotate` |
| Медленный поиск | Seq Scan в `explain()` | `db_index=True` |
| `icontains` не работает для кириллицы | Тест руками | `Lower()` + `contains` |

### Общий алгоритм оптимизации

1. Открыть страницу в браузере с запущенным Debug Toolbar.
2. Посмотреть вкладку SQL: сколько запросов, есть ли дублирующиеся.
3. Найти view, которая генерирует эти запросы.
4. Добавить `select_related` / `prefetch_related` / `annotate`.
5. Обновить страницу, убедиться что количество запросов уменьшилось.
6. Для медленных запросов — проверить `explain()`, добавить индекс если нужно.

### Что не стоит оптимизировать

Не нужно добавлять `select_related` везде подряд. Если связанный объект в шаблоне не используется, JOIN только добавит нагрузку. Оптимизируйте только то, что реально тормозит — и только после того, как измерили.

## 9. Самостоятельные задания

Все задания выполняются в проекте `bookclub`. Запустите сервер перед началом:

```bash
cd /path/to/bookclub
poetry run python manage.py runserver
```


Репозиторий проекта: https://github.com/vitaly-efremov/django-bookclub

---

### Задание 1. N+1 в профиле участника

В `clubs/views.py` функция `member_profile()` загружает `ReadingList` без `select_related`:

```python
def member_profile(request, username):
    member = get_object_or_404(Member, user__username=username)

    # N+1: для каждого entry загружается book отдельным запросом
    reading_list = ReadingList.objects.filter(member=member)

    reviews = Review.objects.filter(member=member)
    ...
```

**Что сделать:**
1. Откройте страницу профиля участника в браузере (создайте участника через `/admin/` если нужно).
2. Посмотрите в Debug Toolbar → SQL сколько запросов выполняется.
3. Исправьте N+1: добавьте `select_related('book')` к `reading_list` и `select_related('book')` к `reviews`.
4. Обновите страницу, убедитесь что количество запросов уменьшилось.

---

### Задание 2. annotate вместо Python-вычислений в api_books()

В `clubs/views.py` функция `api_books()` для каждой книги вызывает `book.average_rating()` (Python-вычисление) и обращается к `book.genre.name` (отдельный запрос):

```python
data = []
for book in books[:20]:
    data.append({
        'id': book.id,
        'title': book.title,
        'author': book.author,
        'year': book.year,
        'genre': book.genre.name if book.genre else None,  # доп. запрос!
        'avg_rating': book.average_rating(),               # ещё запросы!
        'cover_url': book.cover_url,
    })
```

**Что сделать:**
1. Перепишите queryset с использованием `select_related('genre')` и `annotate(avg=Avg('reviews__rating'))`.
2. В цикле замените `book.average_rating()` на `book.avg`, а `book.genre.name` будет работать без доп. запросов благодаря `select_related`.
3. Проверьте через `connection.queries` или Debug Toolbar, что количество запросов к БД теперь не зависит от количества книг.

Подсказка: начало исправленного кода:

```python
from django.db.models import Avg

books = Book.objects.select_related('genre').annotate(avg=Avg('reviews__rating'))
```

---

### Задание 3. Индекс на Book.title

Сейчас поле `Book.title` определено без индекса (намеренно, для демонстрации):

```python
class Book(models.Model):
    # Намеренно НЕТ db_index=True — для демонстрации медленного поиска
    title = models.CharField('Название', max_length=300)
```

**Что сделать:**
1. Убедитесь, что `explain()` показывает `Seq Scan`:
   ```python
   # В Django shell
   from clubs.models import Book
   print(Book.objects.filter(title__icontains='война').explain(analyze=True))
   ```
2. Добавьте `db_index=True` на поле `title` в `clubs/models.py`.
3. Создайте и примените миграцию:
   ```bash
   poetry run python manage.py makemigrations
   poetry run python manage.py migrate
   ```
4. Снова запустите `explain()` и сравните план до и после.